# Genetic Algorithm for WEC Design and Array Layout Optimization

---

## Overview

This notebook implements **Genetic Algorithm (GA)** optimization for wave energy converters (WECs) in two parts:

### Part 1: Single WEC Design Optimization
Optimizes the geometric parameters `[radius, draft]` of an individual WEC to maximize the **power-to-cost ratio**. This demonstrates the basic GA operators (selection, crossover, mutation) on a simple 2-variable problem.

### Part 2: WEC Array Layout Optimization
Optimizes the `(x, y)` positions of $N$ WECs within a rectangular deployment area. The fitness function accounts for:
- **Hydrodynamic interactions** between devices via $q$-factors (Parks, 1976)
- **Minimum spacing constraints** to prevent mooring conflicts and ensure safe operation

### What is a Genetic Algorithm?

A GA is a **metaheuristic optimization** method inspired by natural selection:

1. **Initialize** a random population of candidate solutions (chromosomes)
2. **Evaluate** the fitness of each individual
3. **Select** the fittest individuals as parents
4. **Crossover** parents to produce offspring
5. **Mutate** offspring with some probability
6. **Repeat** for many generations until convergence

GAs are well-suited for WEC array optimization because the search space is non-convex, multimodal, and has many local optima due to oscillatory hydrodynamic interactions.

### Dependencies

```
numpy, scipy
```

In [ ]:
import numpy as np
from scipy.special import j0   # Bessel function of the first kind, order 0

%matplotlib inline
np.random.seed(42)

---

# Part 1: Single WEC Design Optimization

## Fitness Function

The fitness of a WEC design `[radius, draft]` is a simplified **power-to-cost ratio**:

$$\text{fitness} = \frac{\sin(r) \cdot \cos(d) + 1.5}{0.5r + 0.8d}$$

where $r$ is the radius and $d$ is the draft. The numerator represents captured wave power (a nonlinear function of geometry), and the denominator represents manufacturing cost (linear in dimensions).

In [ ]:
def calculate_fitness(individual):
    """Single WEC fitness: simplified power-to-cost ratio."""
    radius, draft = individual
    wave_power_coeff = np.sin(radius) * np.cos(draft) + 1.5
    cost_factor = radius * 0.5 + draft * 0.8
    fitness = wave_power_coeff / cost_factor
    return fitness

## GA Operators

### Initialization
Random population within parameter bounds.

### Selection
**Truncation selection**: keep the top-performing fraction of the population as parents.

### Crossover
**Single-point crossover**: each offspring inherits the first half of genes from one parent and the second half from another.

### Mutation
**Uniform random mutation**: each gene has a probability `mutation_rate` of being replaced by a random value within bounds.

In [ ]:
def create_initial_population(pop_size, bounds):
    """Create random population within parameter bounds."""
    num_vars = bounds.shape[1]
    population = np.empty((pop_size, num_vars))
    for j in range(num_vars):
        population[:, j] = np.random.uniform(bounds[0, j], bounds[1, j], size=pop_size)
    return population


def select_parents(population, fitness_scores, num_parents):
    """Truncation selection: keep top individuals."""
    sorted_indices = np.argsort(fitness_scores)[::-1]
    return population[sorted_indices[:num_parents]]


def crossover(parents, offspring_size):
    """Single-point crossover."""
    offspring = np.empty(offspring_size)
    crossover_point = offspring_size[1] // 2
    num_parents = parents.shape[0]
    for k in range(offspring_size[0]):
        p1 = k % num_parents
        p2 = (k + 1) % num_parents
        offspring[k, :crossover_point] = parents[p1, :crossover_point]
        offspring[k, crossover_point:] = parents[p2, crossover_point:]
    return offspring


def mutate(offspring, mutation_rate, bounds):
    """Uniform random mutation within bounds."""
    for i in range(offspring.shape[0]):
        for j in range(offspring.shape[1]):
            if np.random.rand() < mutation_rate:
                offspring[i, j] = np.random.uniform(bounds[0, j], bounds[1, j])
    return offspring

## Run Single WEC GA

In [ ]:
def run_single_wec_ga():
    """Run GA for single WEC design optimization."""
    pop_size = 50
    num_generations = 100
    mutation_rate = 0.1
    bounds = np.array([[1.0, 1.0], [10.0, 10.0]])  # [radius, draft]

    population = create_initial_population(pop_size, bounds)

    for gen in range(num_generations):
        fitness_scores = np.array([calculate_fitness(ind) for ind in population])
        num_parents = pop_size // 5
        parents = select_parents(population, fitness_scores, num_parents)
        offspring_size = (pop_size - num_parents, parents.shape[1])
        offspring = crossover(parents, offspring_size)
        offspring = mutate(offspring, mutation_rate, bounds)
        population[:num_parents] = parents
        population[num_parents:] = offspring

    fitness_scores = np.array([calculate_fitness(ind) for ind in population])
    best_idx = np.argmax(fitness_scores)
    return population[best_idx], fitness_scores[best_idx]


print("=" * 60)
print("PART 1: Single WEC Design Optimization")
print("=" * 60)
best_design, best_fit = run_single_wec_ga()
print(f"  Best design:  radius = {best_design[0]:.4f},  draft = {best_design[1]:.4f}")
print(f"  Best fitness: {best_fit:.4f}")

---

# Part 2: WEC Array Layout Optimization

## Hydrodynamic Interaction: q-Factor Model

When multiple WECs are deployed in an array, they interact through scattered and radiated waves. The **$q$-factor** quantifies this interaction using a far-field approximation (Parks, 1976):

$$q(d, k) = 1 + \alpha \cdot J_0(kd) \cdot e^{-\gamma kd}$$

where:
- $d$ = separation distance between two WECs (m)
- $k = 2\pi/\lambda$ = wave number (rad/m)
- $J_0$ = Bessel function of the first kind, order 0
- $\alpha$ = interaction strength coefficient
- $\gamma$ = exponential decay rate

**Physical interpretation:**
- $q > 1$: **constructive** interference (power gain due to wave focusing)
- $q < 1$: **destructive** interference (power loss due to wave shadowing)
- $q \to 1$: negligible interaction at large separations

The oscillatory $J_0(kd)$ term creates alternating bands of constructive and destructive interference as a function of spacing, making the optimization landscape highly multimodal.

In [ ]:
def compute_q_factor(distance, wavenumber, alpha=0.8, decay=0.05):
    """
    Hydrodynamic interaction factor between two WECs (far-field approximation).
    """
    if distance < 1e-6:
        return 1.0
    kd = wavenumber * distance
    q = 1.0 + alpha * j0(kd) * np.exp(-decay * kd)
    return max(q, 0.1)

### Visualize the q-Factor

Let's plot $q$ as a function of inter-device spacing to understand the interaction pattern.

In [ ]:
import matplotlib.pyplot as plt
import os

# Locate <repo_root>/figs/04_optimization/standalone/ for this tutorial
# --- bootstrap: locate the repository and make code/ importable ---
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO_ROOT, 'code')):
    if os.path.dirname(REPO_ROOT) == REPO_ROOT:
        raise FileNotFoundError('Repository root not found (no code/ directory). '
                                'Start Jupyter from inside the repository.')
    REPO_ROOT = os.path.dirname(REPO_ROOT)
CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# --- end bootstrap ---
FIGS_DIR = os.path.join(REPO_ROOT, 'figs', '04_optimization', 'standalone')
os.makedirs(FIGS_DIR, exist_ok=True)


wave_period = 8.0  # s
wavelength = 1.56 * wave_period ** 2  # deep-water dispersion
k = 2 * np.pi / wavelength

distances = np.linspace(10, 1000, 500)
q_values = [compute_q_factor(d, k) for d in distances]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distances, q_values, 'b-', lw=1.5)
ax.axhline(y=1.0, color='grey', ls='--', lw=0.8, label='No interaction (q=1)')
ax.fill_between(distances, 1.0, q_values, where=np.array(q_values) > 1,
                alpha=0.2, color='green', label='Constructive (q>1)')
ax.fill_between(distances, 1.0, q_values, where=np.array(q_values) < 1,
                alpha=0.2, color='red', label='Destructive (q<1)')
ax.set_xlabel('Inter-device Spacing (m)', fontsize=11)
ax.set_ylabel('q-factor', fontsize=11)
ax.set_title(f'Hydrodynamic Interaction Factor (Tp={wave_period}s, lambda={wavelength:.0f}m)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, 'ga_array_qfactor_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Array Power and Spacing Penalty

For each WEC $i$ in the array, the effective power is:

$$P_i = P_{\text{isolated}} \cdot \frac{1}{N-1} \sum_{j \neq i} q(d_{ij}, k)$$

The **array interaction factor** $q_{\text{array}} = \sum P_i / (N \cdot P_{\text{isolated}})$ indicates overall gain ($>1$) or loss ($<1$).

A **quadratic penalty** discourages WEC pairs closer than the minimum spacing threshold.

In [ ]:
def compute_array_power(positions, wavenumber, isolated_power):
    """Total array power considering pairwise q-factor interactions."""
    n = positions.shape[0]
    wec_powers = np.zeros(n)

    for i in range(n):
        if n == 1:
            wec_powers[i] = isolated_power
        else:
            q_sum = sum(
                compute_q_factor(np.linalg.norm(positions[i] - positions[j]), wavenumber)
                for j in range(n) if j != i
            )
            wec_powers[i] = isolated_power * (q_sum / (n - 1))

    total_power = wec_powers.sum()
    q_array = total_power / (n * isolated_power)
    return total_power, q_array


def spacing_penalty(positions, min_spacing):
    """Quadratic penalty for WEC pairs closer than min_spacing."""
    n = positions.shape[0]
    penalty = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.linalg.norm(positions[i] - positions[j])
            if dist < min_spacing:
                penalty += (min_spacing - dist) ** 2
    return penalty


def array_fitness(chromosome, n_wecs, wavenumber, isolated_power,
                  min_spacing, penalty_weight=1e4):
    """Fitness = array power - penalty for spacing violations."""
    positions = chromosome.reshape(n_wecs, 2)
    power, _ = compute_array_power(positions, wavenumber, isolated_power)
    pen = spacing_penalty(positions, min_spacing)
    return power - penalty_weight * pen

## Run Array Layout GA

The GA operates on chromosomes of length $2N$ (flattened `[x1, y1, x2, y2, ..., xN, yN]`). It uses **uniform crossover** (each gene independently inherited from either parent) and the same mutation/selection as Part 1.

In [ ]:
def run_array_ga(n_wecs=5, pop_size=100, num_generations=200,
                 mutation_rate=0.15, domain_size=(1000, 500),
                 min_spacing=100, wave_period=8.0, isolated_power=100.0):
    """
    GA for optimizing WEC array layout within a rectangular domain.
    """
    wavelength = 1.56 * wave_period ** 2
    wavenumber = 2 * np.pi / wavelength

    n_vars = 2 * n_wecs
    x_max, y_max = domain_size

    lo = np.tile([0.0, 0.0], n_wecs)
    hi = np.tile([x_max, y_max], n_wecs)

    population = np.empty((pop_size, n_vars))
    for j in range(n_vars):
        population[:, j] = np.random.uniform(lo[j], hi[j], size=pop_size)

    history = np.zeros(num_generations)

    for gen in range(num_generations):
        scores = np.array([
            array_fitness(ind, n_wecs, wavenumber, isolated_power, min_spacing)
            for ind in population
        ])
        history[gen] = scores.max()

        n_parents = max(pop_size // 5, 2)
        parents = population[np.argsort(scores)[::-1][:n_parents]].copy()

        n_offspring = pop_size - n_parents
        offspring = np.empty((n_offspring, n_vars))
        for k in range(n_offspring):
            p1 = parents[k % n_parents]
            p2 = parents[(k + 1) % n_parents]
            mask = np.random.rand(n_vars) < 0.5
            offspring[k] = np.where(mask, p1, p2)

        for i in range(n_offspring):
            for j in range(n_vars):
                if np.random.rand() < mutation_rate:
                    offspring[i, j] = np.random.uniform(lo[j], hi[j])

        population[:n_parents] = parents
        population[n_parents:] = offspring

    scores = np.array([
        array_fitness(ind, n_wecs, wavenumber, isolated_power, min_spacing)
        for ind in population
    ])
    best_idx = np.argmax(scores)
    best_pos = population[best_idx].reshape(n_wecs, 2)
    _, q_arr = compute_array_power(best_pos, wavenumber, isolated_power)

    return best_pos, scores[best_idx], q_arr, history

In [ ]:
print("=" * 60)
print("PART 2: WEC Array Layout Optimization")
print("=" * 60)
positions, fitness, q, hist = run_array_ga(
    n_wecs=5, pop_size=100, num_generations=200,
    domain_size=(1000, 500), min_spacing=100,
    wave_period=8.0, isolated_power=100.0
)
print(f"  WECs in array : {positions.shape[0]}")
print(f"  Best fitness  : {fitness:.2f} kW")
print(f"  Array q-factor: {q:.4f}")
print(f"  Optimal positions:")
for i, (x, y) in enumerate(positions):
    print(f"    WEC {i+1}: ({x:7.1f}, {y:7.1f}) m")

dists = []
for i in range(len(positions)):
    for j in range(i + 1, len(positions)):
        dists.append(np.linalg.norm(positions[i] - positions[j]))
print(f"  Min spacing   : {min(dists):.1f} m")

## Visualization: Array Layout and GA Convergence

In [ ]:
from matplotlib.patches import Rectangle

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Layout plot
ax1.add_patch(Rectangle((0, 0), 1000, 500, lw=1.5, ec='grey',
                         fc='lightyellow', ls='--'))
ax1.scatter(positions[:, 0], positions[:, 1], s=250, c='dodgerblue',
            edgecolors='navy', linewidths=1.5, zorder=5, label='WEC')
for i, (x, y) in enumerate(positions):
    ax1.annotate(f' {i+1}', (x, y), fontsize=10, fontweight='bold')
ax1.set_xlim(-50, 1050); ax1.set_ylim(-50, 550)
ax1.set_aspect('equal')
ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_title(f'Optimal Array Layout ({len(positions)} WECs)\n'
              f'q={q:.3f}, min spacing={min(dists):.0f}m')
ax1.legend(); ax1.grid(True, alpha=.3)

# Convergence plot
ax2.plot(hist, color='darkblue', lw=1.2)
ax2.set_xlabel('Generation'); ax2.set_ylabel('Best Fitness (kW)')
ax2.set_title('GA Convergence')
ax2.grid(True, alpha=.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, 'ga_array_optimal_layout.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Summary

This notebook demonstrated:

1. **GA fundamentals**: population initialization, selection, crossover, mutation
2. **Single WEC design optimization**: maximizing power-to-cost ratio over geometric parameters
3. **Array layout optimization**: placing multiple WECs to maximize total power while respecting spacing constraints
4. **q-factor interaction model**: Bessel-function-based far-field approximation of hydrodynamic interactions

### Key Result

The GA finds WEC positions that exploit constructive interference bands ($q > 1$) while maintaining safe inter-device distances. The array $q$-factor indicates whether the layout outperforms ($q > 1$) or underperforms ($q < 1$) compared to isolated operation.

### Next Steps

In `swan_coupled_ga_array.ipynb`, we replace the uniform wave field assumption with a **SWAN-resolved 2-D wave field**, giving spatially varying wave power across the deployment area.